# Transformer baseline and character-level comparison

Two questions, each answered against a control trained in the same run:

| configuration | question it answers |
|---|---|
| BiLSTM, BPE-1000 pieces only | reference: the paper's headline setting, re-run on this hardware |
| BiLSTM, characters only | do characters work as well as BPE pieces? |
| BiLSTM, word + characters | the Lample et al. (2016) design (optional) |
| BiLSTM, word only | control for the transformer at word level |
| Transformer, word only | the course's required transformer baseline |
| Transformer, BPE-1000 pieces only | does the subword channel help a transformer too? |

Protocol, identical for every row: train on the 6,000-tweet training part,
early-stop on the 1,500 validation tweets (up to 60 epochs, patience 12),
batch 32, CRF on, then score the test split. Five seeds each. This is the
protocol of `notebooks/07_subword_model.py --final`, which produced the earlier
step7b results.

## How to run

1. Settings: Accelerator = GPU T4 x2, Internet = On
2. Leave `SMOKE = True` and Run All (about 10 minutes). Every configuration runs
   for one seed and two epochs, and the last cell projects the full runtime.
3. Set `SMOKE = False`, choose `PART`, then Save Version -> Save & Run All
   (Commit). Collect `/kaggle/working/extra_baselines/` when it finishes.

## Settings

In [ ]:
SMOKE  = True       # True = quick check of every step plus a runtime projection
PART   = 'all'      # 'all', 'char' or 'transformer'
INCLUDE_LAMPLE = False   # also run word + characters (the slowest configuration)

BRANCH = 'sabith'   # the code for these runs lives here until merged into main
REPO   = 'https://github.com/hatheem-r/project_DNN.git'

print('SMOKE TEST' if SMOKE else 'FULL RUN', '| part', PART, '| Lample', INCLUDE_LAMPLE)

## GPU

In [ ]:
import torch, os
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'

## Code

In [ ]:
import os, shutil, subprocess
os.chdir('/kaggle/working')
if os.path.exists('project'):
    shutil.rmtree('project')
!git clone -q -b $BRANCH $REPO project
os.chdir('/kaggle/working/project')
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
os.makedirs('results', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)
helptext = subprocess.run(['python', 'notebooks/07_subword_model.py', '--help'],
                          capture_output=True, text=True).stdout
assert '--encoder' in helptext and 'char' in helptext, 'clone is stale: new options missing'
print('HEAD:', subprocess.run(['git', 'log', '-1', '--oneline'],
                              capture_output=True, text=True).stdout.strip())

## fastText vectors and the embedding matrix

About 460 MB, downloaded to `/kaggle/temp` so it is not saved as output. The
BPE-only and character-only models never use these vectors, but the script
loads the matrix for every configuration, as in the earlier runs, so that model
construction (and therefore the random-number sequence) matches them.

In [ ]:
import os
os.makedirs('/kaggle/temp/embeddings', exist_ok=True)
VEC = '/kaggle/temp/embeddings/cc.si.300.vec.gz'
if not os.path.exists(VEC):
    !wget -q -O $VEC https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
os.environ['SOLD_VECTORS'] = VEC
!python notebooks/03_embeddings.py 2>&1 | tail -4
assert os.path.exists('artifacts/embedding_matrix.npy'), 'embedding matrix was not built'

## BPE-1000 tokenizer

Trained on the 6,000-tweet training part with the same call that
`notebooks/06_subword_tokenizer.py` uses for this setting.

In [ ]:
import sys
sys.path.insert(0, 'src')
from data import load_sold, train_val_split
from subword import train_sentencepiece

train_part, _ = train_val_split(load_sold('train'))
if not os.path.exists('artifacts/sp_bpe_1000.model'):
    train_sentencepiece([' '.join(t) for t in train_part['token_list']], 1000,
                        'artifacts/sp_bpe_1000', model_type='bpe')
print('tokenizer ready:', os.path.exists('artifacts/sp_bpe_1000.model'))

## Tests

The alignment tests check that piece lists line up with words. A break there
shifts labels against tokens without raising an error.

In [ ]:
!python tests/test_metrics.py | tail -2
!python tests/test_subword_alignment.py | tail -2

## Configurations and helper

`run()` streams each command's output live and saves it to a log file.

In [ ]:
import os, subprocess

CONFIGS = [
    # name,              parts,                    --sp,       extra flags
    ('bilstm_bpe_only',  {'char', 'transformer'},  'bpe_1000', ['--no-word-channel']),
    ('bilstm_char_only', {'char'},                 'char',     ['--no-word-channel']),
    ('bilstm_word_char', {'lample'},               'char',     []),
    ('bilstm_word_only', {'transformer'},          'none',     []),
    ('trf_word_only',    {'transformer'},          'none',     ['--encoder', 'transformer']),
    ('trf_bpe_only',     {'transformer'},          'bpe_1000', ['--no-word-channel',
                                                                '--encoder', 'transformer']),
]
wanted = {'char', 'transformer'} if PART == 'all' else {PART}
if INCLUDE_LAMPLE:
    wanted.add('lample')
RUNS = [c for c in CONFIGS if c[1] & wanted]

CSV = 'results/results_extra_smoke.csv' if SMOKE else 'results/results_extra_baselines.csv'
if SMOKE and os.path.exists(CSV):
    os.remove(CSV)
OUT = '/kaggle/working/extra_baselines'
os.makedirs(OUT, exist_ok=True)


def run(args, logfile):
    print('$', ' '.join(args), flush=True)
    with open(logfile, 'w') as log:
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        rc = p.wait()
    print('exit code', rc)
    assert rc == 0, f'failed - see {logfile}'


print('will run:', [c[0] for c in RUNS])

## Runs

Each configuration's log and the results CSV are copied to the output folder
after every run, so a run that stops early keeps what finished.

In [ ]:
import shutil
for name, _, sp, flags in RUNS:
    tag = ('smoke_' if SMOKE else '') + name
    args = ['python', 'notebooks/07_subword_model.py', '--final', '--sp', sp,
            *flags, '--tag', tag, '--csv', CSV]
    if SMOKE:
        args += ['--seeds', '1', '--epochs', '2', '--patience', '2']
    log = f'results/extra_{tag}.txt'
    run(args, log)
    if not SMOKE:
        shutil.copy(log, OUT)
        shutil.copy(CSV, OUT)
    print()

## Results

In smoke mode this projects the full runtime from the measured time per epoch.
Otherwise it aggregates the five seeds and answers each question against its
control: a difference larger than twice the larger standard deviation is
reported as clear, larger than one as modest, and anything smaller as no
measurable difference.

In [ ]:
import pandas as pd

df = pd.read_csv(CSV)
df['name'] = df['tag'].str.replace('smoke_', '', regex=False)

if SMOKE:
    print('SMOKE TEST PASSED - every configuration ran. Scores are meaningless.\n')
    print('Projected full runtime (5 seeds, about 40 epochs each with early stopping):')
    total = 0
    for name, g in df.groupby('name', sort=False):
        per_epoch = (g['seconds'] / g['epochs_run']).mean()
        est = per_epoch * 40 * 5 / 60
        total += est
        print(f'  {name:<18} {per_epoch:6.1f} s/epoch  ->  ~{est:4.0f} min')
    print(f'  {"TOTAL":<18} {"":>13}      ~{total:4.0f} min  (~{total / 60:.1f} h)')
    print('\nKaggle stops a run at 12 hours. If the total is close to that, run')
    print("PART = 'char' and PART = 'transformer' as two separate commits.")
    print('\nNow set SMOKE = False and use Save Version -> Save & Run All.')
else:
    agg = df.groupby('name').agg(
        seeds=('seed', 'count'),
        val=('val_f1', 'mean'), val_sd=('val_f1', 'std'),
        test=('test_f1', 'mean'), test_sd=('test_f1', 'std'),
        P=('test_p', 'mean'), R=('test_r', 'mean'))
    print(agg.round(4).to_string(), '\n')
    print('Reference, earlier hardware (step7b): BiLSTM BPE-1000 only, '
          'val 0.7071 +/- 0.0031, test 0.7059 +/- 0.0040\n')

    def compare(question, a, b):
        if a not in agg.index or b not in agg.index:
            return
        d = agg.loc[a, 'test'] - agg.loc[b, 'test']
        noise = max(agg.loc[a, 'test_sd'], agg.loc[b, 'test_sd'], 1e-4)
        if d > 2 * noise:    verdict = 'CLEARLY HIGHER'
        elif d > noise:      verdict = 'modestly higher'
        elif d > -noise:     verdict = 'NO MEASURABLE DIFFERENCE'
        elif d > -2 * noise: verdict = 'modestly lower'
        else:                verdict = 'CLEARLY LOWER'
        print(f'{question}')
        print(f'  {a} {agg.loc[a, "test"]:.4f} vs {b} {agg.loc[b, "test"]:.4f}: '
              f'{d:+.4f} (larger std {noise:.4f}) -> {verdict}\n')

    compare('Characters vs BPE pieces (BiLSTM, no word channel):',
            'bilstm_char_only', 'bilstm_bpe_only')
    compare('Transformer vs BiLSTM at word level:',
            'trf_word_only', 'bilstm_word_only')
    compare('Does the subword channel help the transformer?',
            'trf_bpe_only', 'trf_word_only')
    compare('Transformer vs BiLSTM with BPE pieces:',
            'trf_bpe_only', 'bilstm_bpe_only')
    compare('Lample design (word + characters) vs characters only:',
            'bilstm_word_char', 'bilstm_char_only')
    shutil.copy(CSV, OUT)
    print('saved to', OUT, ':', sorted(os.listdir(OUT)))

## What to commit

Everything in `extra_baselines/` goes under `results/` in the repo:

```
git add results/extra_*.txt results/results_extra_baselines.csv
git commit -m "Transformer baseline and character-level comparison"
git pull --rebase
git push
```